In [ ]:
from datascience import *
%matplotlib inline

import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
import numpy as np
import warnings

# 3 Door (Classic) Version

In [ ]:
three_door_prizes = ['car', 'goat 1', 'goat 2']
def hide(prizes):
    prize_table = Table().with_column('Prize', prizes)
    return (prize_table.sample(with_replacement=False)
                       .with_column('Door', np.arange(1, len(prizes) + 1))
                       .select('Door', 'Prize'))
hide(three_door_prizes)

In [ ]:
def pick(t):
    """Return the door number of a randomly selected row of a doors table."""
    return np.random.choice(t.column('Door'))

def monty_hall():
    doors = hide(three_door_prizes)
    contestant_initial_choice = pick(doors)
    remaining = doors.where('Door', are.not_equal_to(contestant_initial_choice))
    revealed = pick(remaining.where('Prize', are.containing('goat')))
    remaining = remaining.where('Door', are.not_equal_to(revealed))
    contestant_final_choice = pick(remaining)
    what_happened = Table().with_columns(
        'Door', [contestant_initial_choice, revealed, contestant_final_choice],
        'Status', ['Picked first', 'Revealed', 'Final choice'])
    return doors.join('Door', what_happened, 'Door')

In [ ]:
monty_hall()

In [ ]:
trials = 10000
cars = 0
for i in np.arange(trials):
    if monty_hall().where('Status', 'Final choice').column('Prize').item(0) == 'car':
        cars = cars + 1
print(100 * cars / trials, 'percent cars')

# 4 Door Version from Spring 2026 Midterm

In [ ]:
four_door_prizes = ['car', 'goat 1', 'goat 2', 'goat 3']
hide(four_door_prizes)

In [ ]:
def monty_hall_4(lock):
    doors = hide(four_door_prizes)
    contestant_initial_choice = pick(doors)
    remaining = doors.where('Door', are.not_equal_to(contestant_initial_choice))
    
    if lock:
        locked = pick(remaining)
        remaining = remaining.where('Door', are.not_equal_to(locked))
    else:
        locked = None

    revealed = pick(remaining.where('Prize', are.containing('goat')))
    remaining = remaining.where('Door', are.not_equal_to(revealed))
    contestant_final_choice = pick(remaining)
    
    if lock:
        ignored = None
    else:
        remaining = remaining.where('Door', are.not_equal_to(contestant_final_choice))
        ignored = pick(remaining)
    
    what_happened = Table().with_columns(
        'Door', [contestant_initial_choice, revealed, contestant_final_choice, 
                 locked, ignored],
        'Status', ['Picked first', 'Revealed', 'Final choice',
                   'Locked', 'Ignored'])
    return doors.join('Door', what_happened, 'Door')

In [ ]:
monty_hall_4(True)

In [ ]:
monty_hall_4(False)

In [ ]:
trials = 10000
lock = True
cars = 0
for i in np.arange(trials):
    if monty_hall_4(lock).where('Status', 'Final choice').column('Prize').item(0) == 'car':
        cars = cars + 1
print(100 * cars / trials, 'percent cars')